<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/Notebooks/Deployment/Ollama/Llama/llama_ollama_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Deployment using Ollama (Llama 3.2)

In this notebook I'm deploying a large language model locally using Ollama and querying it through an OpenAI-compatible API — both via curl and Python.

**Model used:** `llama3.2:1b` — made by Meta, 1 billion parameters, around 1.3 GB. Picked this one because it's small enough to run on Colab's free T4 GPU without running out of memory.

**Why Ollama?** It lets you run open-source LLMs locally without needing an API key or paying for cloud inference. It also exposes the same API format as OpenAI, so any code written for ChatGPT works here with just a URL change.

## Installing Ollama

The Ollama installer needs `zstd` (a compression tool) to unpack itself — without it the install fails. So we install that first, then run the official Ollama install script.

In [ ]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Starting the Ollama Server

Ollama works like a local web server — it runs in the background and listens for requests on ports. We start it using `subprocess.Popen` so it doesn't block the notebook, then wait in a loop until it's actually ready to accept connections.

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


## Pulling the Model

This downloads `llama3.2:1b` from Ollama's model library — similar to how you'd pull a Docker image. Takes a couple of minutes the first time. Once it's downloaded it stays cached so you don't have to re-download it in the same session.

In [ ]:
!ollama pull llama3.2:1b

## Querying via curl

Ollama exposes a `/v1/chat/completions` endpoint that follows the exact same format as OpenAI's API. Here I'm using curl to send a POST request with a message and getting the response back as JSON.

The `Authorization: Bearer ollama` header is required by the format but Ollama doesn't actually validate it — any string works.

In [ ]:
%%bash
curl -s http://localhost:11434/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ollama" \
  -d '{
    "model": "llama3.2:1b",
    "messages": [
      {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
  }' | python3 -c "import json,sys; print(json.load(sys.stdin)['choices'][0]['message']['content'])"

A Large Language Model (LLM) is a type of artificial intelligence (AI) designed to process and understand natural language input, such as text or speech. It's like a super-smart internet search engine that can:

* Understand complex sentences
* Translate languages
* Generate human-like responses
* Learn from vast amounts of data

Think of it like a digital conversational AI assistant that can communicate with humans in a way that feels natural and intuitive.


## Querying via Python (OpenAI SDK)

Instead of writing raw HTTP requests, we can use the official `openai` Python library. The only change from normal OpenAI usage is pointing `base_url` to our local Ollama server. Everything else — the method names, parameters, response format — stays identical.

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
)

print(response.choices[0].message.content)

A Large Language Model (LLM) is a type of artificial intelligence (AI) that uses natural language processing (NLP) to analyze, understand, and generate human-like text. It's trained on vast amounts of data, including books, articles, and conversations, allowing it to learn patterns and relationships in language. LLMs can then be used for tasks such as:

* Text generation
* Sentiment analysis
* Question answering
* Translation

Some popular examples of LLMs include Google's CLIO and Microsoft's BERT model.


## Streaming Response

Instead of waiting for the full response to generate before printing, streaming sends tokens back as they're produced — the same way ChatGPT types out answers word by word. Setting `stream=True` enables this.

In [ ]:
stream = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "Explain how Ollama works in simple terms."}
    ],
    stream=True
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

I'd like to help you with that, but I think there might be some confusion. After a quick search, I found that "Olla" is actually the Spanish word for "pot," and "ollama" seems to be an informal or colloquial term for a type of cooking pot.

That being said, Ollamas are likely cooking pots made from ceramic, clay, or other types of earthy materials. Here's how they typically work:

1. Cooking: When you heat the olla on a stovetop or in the oven, it gets hot enough to bring out the flavors and moisture from the food inside.
2. Stovetop use: On a stovetop, the olla is typically left simmering for several hours while you cook other dishes.
3. Oven use: When you need food cooked quickly, you can put the olla in the oven at high heat to cook it faster.

When combined with water and cooking time, Ollamas can be quite effective at slow-cooking stews, soups, and other hearty meals that simmer for several hours.

If this isn't what you were thinking of or if you'd like more information on cookin

## Multi-turn Conversation

LLMs don't have any memory on their own — every request is independent. To simulate a conversation, we manually keep track of the chat history and send the full message list with every new request. That way the model has context from previous turns.

The `chat()` function below handles this — it appends the user message, gets a reply, then appends that reply too so the next call includes the full history.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})
    res = client.chat.completions.create(model="llama3.2:1b", messages=messages)
    reply = res.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is the transformer architecture?"))
print(chat("What role does attention play in it?"))

The Transformer architecture is a neural network designed for machine translation and other natural language processing tasks. It was introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al.

In traditional recurrent neural networks (RNNs), such as Long Short-Term Memory (LSTM) cells, the input sequence is processed sequentially, one element at a time. This can lead to long dependencies between adjacent elements in the sequence.

The Transformer architecture addresses this problem by introducing an auto-regressive encoder-de decoder model with self-attention mechanisms. Here's a high-level overview:

**Encoder:**

1. Input sequence (x) of input words or tokens
2. Embeddings for each word, using a fixed-size vector (e.g., 300 dimensions)
3. Positional encoding to represent the absolute position in the sequence, similar to RNNs

The encoder processes the input sequence and extracts vector representations for each token (word). These vectors are then combined to form a c